## Import packages 

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

## Set up base paths and paths to read in files

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
land_use = base_path / "Inputs/2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

## Set up output folder 

In [ ]:
# Define the output folder path
output_folder = base_path / "Processed_data/existing_future_forest"

In [ ]:
map_forest_connectivity_figures_folder = base_path / "Results/Map_Figures/forest_connectivity"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
# Reproject to Jamaica Metric Grid (EPSG:3448)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Landcover CRS:", terrestrial_landcover.crs)

## Define which classes count as "Plantation" or "Bamboo"

In [ ]:
plantation_and_bamboo_classes = [
    'Bamboo and Fields',
    'Fields  and Bamboo',
    'Bamboo',
    'Bamboo and Secondary Forest',
    'Plantation: Tree crops, shrub crops, sugar cane, banana',
    'Hardwood Plantation: Euculytus',
    'Hardwood Plantation: Mixed',
    'Hardwood Plantation: Mahoe',
    'Hardwood Plantation: Mahogany', 
]

## Filter plantation or bamboo

In [ ]:
plantation_and_bamboo_classes = terrestrial_landcover[terrestrial_landcover["Classify"].isin(plantation_and_bamboo_classes)].copy()
print(f"Number of forest polygons: {len(plantation_and_bamboo_classes)}")


In [ ]:
# Calculate area for each polygon (in m²) and then group by class
plantation_and_bamboo_classes['area_m2'] = plantation_and_bamboo_classes.geometry.area
area_by_class = plantation_and_bamboo_classes.groupby('Classify')['area_m2'].sum()



In [ ]:
# Define a color mapping. For the mixed classes, choose a base color representing the dominant component.
color_dict = {
    'Bamboo and Fields': '#F4A460',             # base: bamboo; hatch will denote agriculture
    'Fields  and Bamboo': '#8B0000',              # base: agriculture; hatch will denote bamboo
    'Bamboo': '#FFA500',                         # pure bamboo – SeaGreen
    'Bamboo and Secondary Forest': '#9ACD32',    # mix – YellowGreen
    'Plantation: Tree crops, shrub crops, sugar cane, banana': '#FFD700',  # Gold
    'Hardwood Plantation: Euculytus': '#708090',   # SlateGray
    'Hardwood Plantation: Mixed': '#A0522D',       # Sienna
    'Hardwood Plantation: Mahoe': '#A52A2A',       # Brown
    'Hardwood Plantation: Mahogany': '#B22222',    # FireBrick
}


In [ ]:
# Define a dictionary for the mixed classes and their descriptions (percentages)
mixed_classes = {
    'Bamboo and Fields': '75% Bamboo, 25% Ag',
    'Fields  and Bamboo': '75% Ag, 25% Bamboo',
    'Bamboo and Secondary Forest': '75% Bamboo, 25% Forest'
}


In [ ]:
# Set up the plot with common extents (using hydrobasins extent for context)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)


In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot hydrobasins as a background layer
hydrobasins.plot(ax=ax, color="white", edgecolor="blue")

# Plot each agricultural class and build legend handles with area info
legend_handles = []
for cls, base_color in color_dict.items():
    subset = plantation_and_bamboo_classes[plantation_and_bamboo_classes["Classify"] == cls]
    if not subset.empty:
        if cls in mixed_classes:
            # For mixed classes, add a hatch pattern
            subset.plot(ax=ax, color=base_color, edgecolor="black", alpha=0.7,
                        hatch='//', zorder=101)
            description = mixed_classes[cls]
            total_area = area_by_class.get(cls, 0)
            total_area_ha = total_area / 10000  # Convert m² to ha
            label = f"{cls} ({description}, {total_area_ha:,.0f} ha)"
            patch = mpatches.Patch(facecolor=base_color, hatch='//', edgecolor="black", label=label)
            legend_handles.append(patch)
        else:
            subset.plot(ax=ax, color=base_color, edgecolor="black", alpha=0.7, zorder=101)
            total_area = area_by_class.get(cls, 0)
            total_area_ha = total_area / 10000  # Convert m² to ha
            label = f"{cls} ({total_area_ha:,.0f} ha)"
            legend_handles.append(mpatches.Patch(color=base_color, label=label))

# Optionally, add a legend entry for the hydrobasins background layer
bg_legend = Line2D([0], [0], color='blue', lw=2, label='Catchments')
legend_handles.insert(0, bg_legend)

# Define scale bar and north arrow functions (as before)
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add the scale bar and north arrow to the map
add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax, location=(0.9, 0.85))

# Set axis limits, labels, and title
ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Plantation and Bamboo", fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# Add the legend (positioned beneath the plot)
ax.legend(handles=legend_handles, title="Plantation and bamboo classes", 
          loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, 
          frameon=False, fontsize=12, title_fontsize=14)

plt.tight_layout()
fig.savefig(map_forest_connectivity_figures_folder / "plantation_and_bamboo_classes.png", dpi=300, bbox_inches="tight")
plt.show()